# Hackathon Judge Fine-tuning with Unsloth + SFT

Fine-tunes a small Qwen3 model to judge hackathon projects using:
- **Dataset**: `twangodev/madhacks25-synthetic-pairwise` — real MadHacks pairwise judgments from Qwen3.5-27B
- **LoRA** for parameter-efficient fine-tuning
- **SFT (Supervised Fine-Tuning)** — trains the model to imitate the frontier model's full reasoning chain and verdict

SFT is cheaper and more stable than GRPO: one forward+backward pass per step vs GRPO's N inference samples per step.
The tradeoff is that SFT clones the frontier model's behavior rather than discovering its own reasoning via RL.

**Runs on:** Free Google Colab T4 (16GB VRAM) or any GPU with 8GB+ VRAM.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth unsloth_zoo trl datasets scikit-learn

## 2. Load the MadHacks Dataset

Each row is one pairwise judgment made by Qwen3.5-27B (the frontier model).
- `messages`: full conversation — system prompt + two project descriptions + frontier model's reasoning + verdict
- `verdict`: frontier model's choice (`A` or `B`) — this is our training label
- `position`: whether projects were shown as `ab` or `ba` (position-swapped pairs for bias checking)
- `gt_a_result` / `gt_b_result`: human award labels for each project

In [ ]:
from datasets import load_dataset
import pandas as pd

raw_ds = load_dataset("twangodev/madhacks25-synthetic-pairwise", split="train")
df = raw_ds.to_pandas()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Verdict distribution: {df['verdict'].value_counts().to_dict()}")
print(f"Unique pair_ids: {df['pair_id'].nunique()} (each pair appears twice with positions swapped)")
print(f"Frontier model: {df['model'].unique().tolist()}")

sample = df.iloc[0]
print(f"\nSample row:")
print(f"  pair_id:  {sample['pair_id']}")
print(f"  position: {sample['position']}")
print(f"  verdict:  {sample['verdict']}")
print(f"  gt_a:     {sample['gt_a_result']}")
print(f"  gt_b:     {sample['gt_b_result']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.parquet:   0%|          | 0.00/98.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset shape: (20, 15)
Columns: ['messages', 'judgment_id', 'pair_id', 'position', 'project_a_id', 'project_b_id', 'verdict', 'gt_a_result', 'gt_b_result', 'model', 'prompt_tokens', 'completion_tokens', 'finish_reason', 'latency_s', 'sampling']
Verdict distribution: {'A': 10, 'B': 10}
Unique pair_ids: 10 (each pair appears twice with positions swapped)
Frontier model: ['Qwen/Qwen3.5-27B']

Sample row:
  pair_id:  aa915028ddf17a64
  position: ba
  verdict:  A
  gt_a:     Winner Best Community Hack
  gt_b:     Winner MadHacks Second Place


## 3. Format Dataset for SFT

SFT needs the full conversation including the assistant turn — the model is trained to reproduce
the frontier model's complete reasoning chain and verdict.

The `messages` column already has the right structure: `[system, user, assistant]`.
We just need to apply the chat template and split by `pair_id`.

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

def format_for_sft(row):
    """
    Keep the full messages including the assistant turn.
    SFT trains the model to reproduce the frontier model's reasoning + verdict.
    """
    return {
        "messages": row["messages"],   # system + user + assistant (full)
        "pair_id": row["pair_id"],
        "position": row["position"],
        "answer": row["verdict"],      # kept for eval
        "gt_a_result": row["gt_a_result"],
        "gt_b_result": row["gt_b_result"],
    }

formatted = [format_for_sft(row) for _, row in df.iterrows()]

# Split by pair_id — keeps both position variants together
unique_pairs = df["pair_id"].unique()
train_pairs, test_pairs = train_test_split(unique_pairs, test_size=0.2, random_state=42)

train_data = [ex for ex in formatted if ex["pair_id"] in train_pairs]
test_data  = [ex for ex in formatted if ex["pair_id"] in test_pairs]

train_dataset = Dataset.from_list(train_data)
test_dataset  = Dataset.from_list(test_data)

print(f"Train: {len(train_dataset)} examples ({len(train_pairs)} unique pairs)")
print(f"Test:  {len(test_dataset)} examples ({len(test_pairs)} unique pairs)")
print(f"\nMessage roles: {[m['role'] for m in train_dataset[0]['messages']]}")
print(f"Frontier verdict (training target): {train_dataset[0]['answer']}")
print(f"\nAssistant response preview:")
assistant_content = train_dataset[0]['messages'][2]['content']
print(assistant_content[:300] + "...")

Train: 16 examples (8 unique pairs)
Test:  4 examples (2 unique pairs)

Prompt roles: ['system', 'user']
Label (frontier verdict): A


## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastModel
import torch

# Qwen3-1.7B fits on a free Colab T4.
# Swap to unsloth/Qwen3-4B if you have more VRAM.
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B",  # smallest Qwen3 for testing
    # model_name="unsloth/Qwen3-1.7B",
    max_seq_length=4096,
    load_in_4bit=True,
    full_finetuning=False,
)

print(f"Model loaded on: {next(model.parameters()).device}")

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/576M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded on: cuda:0


## 5. Apply LoRA Adapters

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Trainable: 4,587,520 / 393,019,392 (1.17%)


## 6. Verdict Parser (for Evaluation)

We still need `parse_verdict` to evaluate the model's outputs at test time.
During SFT training this isn't used — the loss is just cross-entropy on the assistant tokens.

In [ ]:
import re

def parse_verdict(response: str):
    """
    Extract VERDICT: A/B/TIE from response.
    Matches the format used in the dataset's system prompt.
    """
    if "</think>" in response:
        response = response.split("</think>")[-1]

    match = re.search(r"VERDICT:\s*([AB]|TIE)", response, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    # Fallback
    if re.search(r"\bProject\s+A\b", response, re.IGNORECASE): return "A"
    if re.search(r"\bProject\s+B\b", response, re.IGNORECASE): return "B"
    return None

# Quick check
print(parse_verdict("<think>Project A is better</think>\nVERDICT: A"))  # A
print(parse_verdict("VERDICT: B"))                                        # B
print(parse_verdict("I cannot decide"))                                   # None

Reward checks:
  Correct:     [1.0]
  Wrong:       [0.0]
  Unparseable: [-0.5]
  Format+:     [0.2]
  Format-:     [0.0]


## 7. Train with SFT

`SFTTrainer` applies the chat template and trains only on the assistant tokens (not the prompt).
This means the model learns to reproduce the frontier model's reasoning and verdict
without wasting loss signal on the system/user turns it will never generate.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./hackathon_judge_sft",
    num_train_epochs=3,              # SFT converges faster than GRPO — 3 epochs is reasonable
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,              # SFT uses a higher LR than GRPO (no RL instability risk)
    warmup_ratio=0.1,
    logging_steps=5,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    # SFTTrainer-specific: only compute loss on assistant tokens
    dataset_text_field=None,         # we're passing messages directly
    max_seq_length=4096,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
)

print(f"Training on {len(train_dataset)} examples ({len(train_pairs)} unique pairs)")
print(f"Method: SFT — cross-entropy loss on assistant tokens only")
print(f"Epochs: {training_args.num_train_epochs}")
print("="*60)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training on 16 examples (8 unique pairs)
Sampling 2 responses per prompt


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16 | Num Epochs = 13 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 4,587,520 of 600,637,440 (0.76% trained)
Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / format_reward / mean,rewards / format_reward / std
10,0.000000,0.745000,0.148492,447.875000,364.300000,545.400000,0.000000,447.875000,364.300000,545.400000,0.000013,0.550000,0.488675,0.195000,0.010000
20,0.000000,0.650000,0.141421,433.325000,363.000000,521.800000,0.000000,433.325000,363.000000,521.800000,0.000086,0.450000,0.315470,0.200000,0.000000
30,0.000000,0.675000,0.176777,440.300000,357.400000,525.000000,0.000000,440.300000,357.400000,525.000000,0.000223,0.475000,0.365470,0.200000,0.000000
40,0.000000,0.725000,0.176777,405.675000,323.800000,498.900000,0.000000,405.675000,323.800000,498.900000,0.000377,0.525000,0.265470,0.200000,0.000000
50,0.000001,0.800000,0.212132,443.575000,321.000000,566.000000,0.000000,443.575000,321.000000,566.000000,0.000528,0.600000,0.415470,0.200000,0.000000
60,0.000001,0.695000,0.077782,460.200000,346.300000,566.000000,0.000000,460.200000,346.300000,566.000000,0.000584,0.500000,0.230940,0.195000,0.010000
70,0.000001,0.650000,0.070711,395.100000,316.400000,473.600000,0.000000,395.100000,316.400000,473.600000,0.000691,0.450000,0.215470,0.200000,0.000000
80,0.000001,0.675000,0.035355,428.275000,337.000000,531.000000,0.000000,428.275000,337.000000,531.000000,0.000672,0.475000,0.338675,0.200000,0.000000
90,0.000001,0.650000,0.141421,458.775000,353.900000,575.800000,0.000000,458.775000,353.900000,575.800000,0.000673,0.450000,0.273205,0.200000,0.000000
100,0.000001,0.775000,0.035355,448.350000,332.500000,544.200000,0.000000,448.350000,332.500000,544.200000,0.000664,0.575000,0.280940,0.200000,0.000000


Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

TrainOutput(global_step=100, training_loss=4.5540548228473197e-07, metrics={'train_runtime': 10115.9218, 'train_samples_per_second': 0.04, 'train_steps_per_second': 0.01, 'total_flos': 0.0, 'train_loss': 4.5540548228473197e-07})

## 8. Save LoRA Weights

In [ ]:
trainer.model.save_pretrained("hackathon_judge_lora")
tokenizer.save_pretrained("hackathon_judge_lora")
print("Saved LoRA adapter to ./hackathon_judge_lora")

Saved.


## 9. Evaluate on Held-out Test Set

Measures agreement with the frontier model's verdict on unseen pairs.

In [ ]:
# Reload base model fresh, then attach the saved adapter
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/Qwen3-0.6B",
    max_seq_length=4096,
    load_in_4bit=True,
)

from peft import PeftModel
model = PeftModel.from_pretrained(model, "hackathon_judge_lora")
model.eval()
print("Loaded.")

def run_inference(prompt_messages):
    text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response, parse_verdict(response)


print("Test Set Evaluation")
print("="*60)

frontier_correct = 0
results = []

for ex in test_dataset:
    response, predicted = run_inference(ex["prompt"])
    frontier_match = predicted == ex["answer"]
    frontier_correct += int(frontier_match)

    think_text = ""
    if "<think>" in response and "</think>" in response:
        think_text = response.split("<think>")[1].split("</think>")[0].strip()

    results.append({
        "pair_id": ex["pair_id"],
        "position": ex["position"],
        "predicted": predicted,
        "frontier_verdict": ex["answer"],
        "gt_a": ex["gt_a_result"],
        "gt_b": ex["gt_b_result"],
        "frontier_match": frontier_match,
    })

    print(f"\npair {ex['pair_id'][:8]} | pos={ex['position']}")
    print(f"  Predicted: {predicted} | Frontier: {ex['answer']} | {'✓' if frontier_match else '✗'}")
    print(f"  GT A: {ex['gt_a_result']}")
    print(f"  GT B: {ex['gt_b_result']}")
    if think_text:
        print(f"  Reasoning: {think_text[:200]}...")

print("\n" + "="*60)
print(f"Agreement with frontier model: {frontier_correct}/{len(test_dataset)} = {frontier_correct/len(test_dataset)*100:.1f}%")

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loaded.
Test Set Evaluation


...


pair 91412e93 | pos=ab
  Predicted: A | Frontier: B | ✗
  GT A: Winner MadHacks Second Place
  GT B: Winner Fish Audio Prize
  Reasoning: Okay, let's see. The user wants me to compare the two projects presented, Factify and Unsilenced, and decide which one is stronger overall. The criteria are originality, technical depth, execution qua...


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



pair 91412e93 | pos=ba
  Predicted: A | Frontier: A | ✓
  GT A: Winner Fish Audio Prize
  GT B: Winner MadHacks Second Place
  Reasoning: Okay, let's compare these two projects. First, I need to check each criteria they have.

Project A has a lot of technology stack: CSS, Face API, Fish Audio, Flask, etc. That's a strong technical depth...


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



pair 906cd078 | pos=ba
  Predicted: A | Frontier: A | ✓
  GT A: Winner Best AI Project
  GT B: Winner MadHacks Second Place
  Reasoning: Okay, let me start by comparing the two projects. First, Project A is GPT Rewind, and Project B is Factify. The user wants to know which one is stronger.

Looking at the originality and creativity, Pr...

pair 906cd078 | pos=ab
  Predicted: A | Frontier: B | ✗
  GT A: Winner MadHacks Second Place
  GT B: Winner Best AI Project
  Reasoning: Okay, let's compare these two projects. First, Project A is "Factify," and Project B is "GPT Rewind." The user wants to know which is stronger. 

Starting with the originality and creativity. Project ...

Agreement with frontier model: 2/4 = 50.0%


## 10. Position Bias Check

Each pair appears twice with projects in opposite order.
A consistent model should pick the same underlying project regardless of which slot it appears in.
Inconsistency = position bias (model favors whichever project appears first).

In [ ]:
results_df = pd.DataFrame(results)

print("Position Bias Analysis")
print("="*60)

n_consistent = 0
n_pairs_checked = 0

for pair_id in test_pairs:
    pair_rows = results_df[results_df["pair_id"] == pair_id]
    if len(pair_rows) < 2:
        continue

    ab_row = pair_rows[pair_rows["position"] == "ab"]
    ba_row = pair_rows[pair_rows["position"] == "ba"]
    if ab_row.empty or ba_row.empty:
        continue

    ab_verdict = ab_row.iloc[0]["predicted"]
    ba_verdict = ba_row.iloc[0]["predicted"]

    # Consistent: same underlying project wins in both orderings
    # ab=A means project_a_id won; ba=B means project_a_id won (since they were swapped)
    consistent = (ab_verdict == "A" and ba_verdict == "B") or \
                 (ab_verdict == "B" and ba_verdict == "A")

    n_consistent += int(consistent)
    n_pairs_checked += 1

    print(f"\npair {pair_id[:8]}:")
    print(f"  GT A: {ab_row.iloc[0]['gt_a']}")
    print(f"  GT B: {ab_row.iloc[0]['gt_b']}")
    print(f"  ab verdict: {ab_verdict} | ba verdict: {ba_verdict}")
    print(f"  Consistent: {'✓' if consistent else '✗ position bias detected'}")

print(f"\nConsistency: {n_consistent}/{n_pairs_checked} pairs = {n_consistent/n_pairs_checked*100:.0f}%")

Position Bias Analysis

pair 906cd078:
  GT A: Winner MadHacks Second Place
  GT B: Winner Best AI Project
  ab verdict: A | ba verdict: A
  Consistent: ✗ position bias detected

pair 91412e93:
  GT A: Winner MadHacks Second Place
  GT B: Winner Fish Audio Prize
  ab verdict: A | ba verdict: A
  Consistent: ✗ position bias detected

Consistency: 0/2 pairs = 0%


## 11. Next Steps

**Scale the data**: Push your full DevPost scrape to HuggingFace in the same schema and swap the `load_dataset` call. Target 5k+ examples.

**Scale training**: Increase `max_steps` to 300-500 with a real dataset, and `num_generations` to 8 for a stronger GRPO signal.

**Larger model**: Swap `Qwen3-1.7B` for `Qwen3-4B` (needs ~10GB VRAM with 4-bit).

**Bradley-Terry ranking**: Convert all pairwise verdicts to a full ranking:

```python
# pip install choix
import choix, numpy as np

# comparisons: list of (winner_idx, loser_idx) tuples
params = choix.lsr_pairwise(n_items=num_projects, data=comparisons)
rankings = np.argsort(params)[::-1]
```

**Human alignment eval**: Compare BT rankings from each model against MadHacks human percentiles to measure which model best replicates human judgment.